In [ ]:
#| default_exp env

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Environment values used to build and ship a project. `EnvStore` loads dockeasy only when a value is read or written.

In [ ]:
#| export
from __future__ import annotations
import asyncio, inspect, os, sys, threading
from fastcore.all import Path

In [ ]:
#| export
BUNDLE_ONLY = ('PYTHONHOME', 'PYTHONPATH', 'PYTHONEXECUTABLE', '__PYVENV_LAUNCHER__', 'RESOURCEPATH')

class EnvError(RuntimeError): pass

`BUNDLE_ONLY` lists interpreter variables inherited from a frozen application. `EnvError` reports invalid or unavailable environment storage.

In [ ]:
#| export
def strip_bundle(env, frozen=None):
    "Return `env` without frozen-host interpreter redirection."
    if not (getattr(sys, 'frozen', False) if frozen is None else frozen): return env
    for name in BUNDLE_ONLY: env.pop(name, None)
    env['PYTHONUTF8'] = '1'
    return env

def clean_env():
    "Return this process's environment without bundle variables."
    return strip_bundle(os.environ.copy(), frozen=True)

`strip_bundle` mutates its mapping. Outside a bundle it returns the mapping unchanged. `clean_env` always removes bundle variables and sets `PYTHONUTF8`.

In [ ]:
env = {'PYTHONHOME': '/Pullup.app/Contents/Resources', 'PATH': '/usr/bin', 'HOME': '/Users/me'}
strip_bundle(dict(env), frozen=True)

{'PATH': '/usr/bin', 'HOME': '/Users/me', 'PYTHONUTF8': '1'}

In [ ]:
#| hide
test_eq(strip_bundle(dict(env), frozen=False), env)
out = strip_bundle(dict(env), frozen=True)
assert not (set(BUNDLE_ONLY) & set(out)), 'every redirection name is gone'
test_eq(out['HOME'], '/Users/me')
same = dict(env)
test_is(strip_bundle(same, frozen=True), same)

In [ ]:
#| export
def venv_env(python=None, env=None):
    "Return `env` with `python`'s virtual environment prepended to `PATH`; None leaves the environment unchanged."
    env = strip_bundle(dict(os.environ if env is None else env))
    if not python: return env
    bindir = str(Path(python).parent)
    env['VIRTUAL_ENV'] = str(Path(bindir).parent)
    env.pop('UV_PROJECT_ENVIRONMENT', None)
    env['PATH'] = bindir + os.pathsep + env.get('PATH', '')
    env.pop('PYTHONHOME', None)
    return env

`venv_env` copies the mapping, sets `VIRTUAL_ENV`, prepends the interpreter directory to `PATH`, and removes `PYTHONHOME` and `UV_PROJECT_ENVIRONMENT`.

In [ ]:
e = venv_env('/repo/.venv/bin/python', env={'PATH': '/usr/bin', 'PYTHONHOME': '/frozen'})
e['VIRTUAL_ENV'], e['PATH'], 'PYTHONHOME' in e

('/repo/.venv', '/repo/.venv/bin:/usr/bin', False)

In [ ]:
#| hide
given = {'PATH': '/usr/bin', 'UV_PROJECT_ENVIRONMENT': '/elsewhere'}
out = venv_env('/repo/.venv/bin/python', env=given)
test_eq(given, {'PATH': '/usr/bin', 'UV_PROJECT_ENVIRONMENT': '/elsewhere'})
assert 'UV_PROJECT_ENVIRONMENT' not in out
test_eq(out['PATH'].split(os.pathsep), ['/repo/.venv/bin', '/usr/bin'])
test_eq(venv_env(None, env={'PATH': '/usr/bin'}), {'PATH': '/usr/bin'})

In [ ]:
#| export
_extra = 'pullup'

def use_extra(name):
    "Name the package every message about the cloud half tells the reader to install."
    global _extra
    _extra = str(name)

def extra(): return _extra

def needs_extra(what): return f'{what}: pip install "{_extra}"'

`use_extra` changes the package name shown by optional-dependency errors.

In [ ]:
#| export
def _dockeasy():
    try: from dockeasy.core import env_get, env_set, secret_get, secret_set
    except ImportError as e:
        raise EnvError(needs_extra('environment values are stored by dockeasy')) from e
    import logging
    logging.getLogger('dotenv.main').setLevel(logging.ERROR)
    return env_get, env_set, secret_get, secret_set

def _call(fn, *args, **kwargs):
    "Call `fn`, awaiting it on its own loop when dockeasy hands back a coroutine."
    value = fn(*args, **kwargs)
    if not inspect.isawaitable(value): return value
    try: asyncio.get_running_loop()
    except RuntimeError: return asyncio.run(value)
    out = []
    def run():
        try: out.append((True, asyncio.run(value)))
        except BaseException as e: out.append((False, e))
    thread = threading.Thread(target=run); thread.start(); thread.join()
    ok, value = out[0]
    if ok: return value
    raise value

`_dockeasy` imports the store on first use. `_call` runs synchronous and asynchronous store functions, including from a running event loop, and re-raises their exceptions.

In [ ]:
async def double(x): return x*2
_call(double, 21), _call(lambda x: x*2, 21)

(42, 42)

In [ ]:
#| hide
async def refused(): raise EnvError('the store said no')
test_fail(lambda: _call(refused), contains='the store said no')
out = []
def no_loop_here(): out.append(_call(double, 21))
t = threading.Thread(target=no_loop_here); t.start(); t.join()
test_eq(out, [42])

In [ ]:
#| hide
test_eq(extra(), 'pullup')
test_eq(needs_extra('this needs gheasy'), 'this needs gheasy: pip install "pullup"')
use_extra('gheasy')
try:
    test_eq(extra(), 'gheasy')
    test_eq(needs_extra('this needs gheasy'), 'this needs gheasy: pip install "gheasy"')
finally: use_extra('pullup[cloud]')
test_eq(needs_extra('x'), 'x: pip install "pullup[cloud]"')

In [ ]:
#| export
class EnvStore:
    "The environment keys one project cares about, read and written through dockeasy."
    def __init__(self, service='fastops', path=None): self.service, self.path = service, path
    def get(self, key, secret=True):
        "One value, from the keychain or the env file, falling back to this process's environment."
        env_get, _set, secret_get, _sset = _dockeasy()
        stored = (_call(secret_get, key, service=self.service, path=self.path) if secret
                  else _call(env_get, key, path=self.path))
        return stored or os.environ.get(key) or ''
    def set(self, key, value, secret=True):
        "Store one value. A secret goes to the keychain as well as the file; a variable does not."
        key = str(key or '').strip()
        if not key: raise EnvError('a key is required')
        if not str(value): raise EnvError(f'{key} needs a value')
        env_get, env_set, secret_get, secret_set = _dockeasy()
        if secret: _call(secret_set, key, str(value), service=self.service, path=self.path)
        else: _call(env_set, key, str(value), path=self.path)
        return {'key': key, 'secret': bool(secret)}
    def unset(self, key):
        "Remove a value from every place `set` put it. Says which places actually held one."
        from dockeasy.core import _FASTOPS_ENV
        from dotenv import unset_key
        key, gone = str(key), []
        try:
            import keyring
            if keyring.get_password(self.service, key) is not None:
                keyring.delete_password(self.service, key)
                gone.append('keychain')
        except Exception: pass
        path = str(self.path or _FASTOPS_ENV)
        try:
            removed, _ = unset_key(path, key)
            if removed: gone.append('env file')
        except Exception: pass
        os.environ.pop(key, None)
        return {'key': key, 'removed': gone}
    def values(self, keys, secret=True):
        "Every key that has a value, as a dict. A key with none is absent rather than empty."
        out = {}
        for k in keys:
            try: v = self.get(k, secret=secret)
            except Exception: v = os.environ.get(k) or ''
            if v: out[k] = v
        return out

`EnvStore` reads and writes a project's environment values. Secrets use the keychain and env file; other values use the env file. Reads fall back to `os.environ`. `values` omits missing keys, and `unset` also removes the process value.

In [ ]:
#| hide
tmp = TemporaryDirectory()
envfile = Path(tmp.name)/'.env'

In [ ]:
store = EnvStore(path=envfile)
store.set('PULLUP_DEMO_REGISTRY', 'ghcr.io/example', secret=False)
store.get('PULLUP_DEMO_REGISTRY', secret=False)

'ghcr.io/example'

These examples use non-secret values. Pass `secret=True` to use the keychain.

In [ ]:
store.values(['PULLUP_DEMO_REGISTRY', 'PULLUP_DEMO_TAG'], secret=False)

{'PULLUP_DEMO_REGISTRY': 'ghcr.io/example'}

In [ ]:
store.unset('PULLUP_DEMO_REGISTRY')

{'key': 'PULLUP_DEMO_REGISTRY', 'removed': ['env file']}

In [ ]:
#| hide
test_eq(store.get('PULLUP_DEMO_REGISTRY', secret=False), '')
test_eq(store.values(['PULLUP_DEMO_REGISTRY'], secret=False), {})
test_fail(lambda: store.set('  ', 'x'), contains='key is required')
test_fail(lambda: store.set('PULLUP_DEMO_TAG', ''), contains='needs a value')

In [ ]:
#| export
def env_value(store, key, default='', secret=False):
    "One value out of an environment store, or `default` when it is unset or unreadable."
    try: return store.get(key, secret=secret) or default
    except Exception: return default

`env_value` returns `default` when a value is missing or unreadable. It reads non-secret values by default.

In [ ]:
env_value(store, 'PULLUP_DEMO_REGISTRY', 'ghcr.io/fallback')

'ghcr.io/fallback'

In [ ]:
#| hide
class Broken:
    def get(self, key, secret=False): raise EnvError('no store here')
test_eq(env_value(Broken(), 'PULLUP_DEMO_REGISTRY', 'ghcr.io/fallback'), 'ghcr.io/fallback')
test_eq(env_value(Broken(), 'PULLUP_DEMO_REGISTRY'), '')

In [ ]:
#| hide
os.environ.pop('PULLUP_DEMO_REGISTRY', None)
tmp.cleanup()